# 1. Mount Google Drive & Setup Workspace

This section mounts your Google Drive to `/content/drive` and changes the current working directory to the folder containing your project files so that all imports, data scripts, and config files work relative to your workspace.

In [ ]:
# ── Constants ──
# Set this path to point to your specific project folder in Google Drive
WORKSPACE_PATH = "/content/drive/MyDrive/Tesi_RAG_Ricette"

from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')

if os.path.exists(WORKSPACE_PATH):
    os.chdir(WORKSPACE_PATH)
    print(f"\nSuccess: Current working directory changed to: {os.getcwd()}")
else:
    print(f"\nERROR: Workspace directory not found at: {WORKSPACE_PATH}")
    print("Please upload the project folder to Google Drive and configure the WORKSPACE_PATH above.")

# 2. Install Dependencies & Start Ollama

Here, we install the required packages (like `sentence-transformers`, `faiss-cpu`, etc.), install `zstd` (required by the Ollama installer to extract files in the Colab container), and set up Ollama.
Since Ollama needs a running background daemon to run locally in Colab, we start it in the background using `subprocess.Popen`.

In [ ]:
# Install Python dependencies
print("Installing python dependencies...")
!pip install -q sentence-transformers faiss-cpu ollama PyYAML pandas tqdm matplotlib seaborn

# Install zstd (required by Ollama installer for extraction)
print("\nInstalling zstd library...")
!apt-get update && apt-get install -y zstd

# Install Ollama CLI/Daemon
print("\nInstalling Ollama CLI/Daemon...")
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import socket

def is_ollama_running():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', 11434)) == 0

if is_ollama_running():
    print("Ollama is already running.")
else:
    print("Starting Ollama background daemon...")
    # Redirect output to prevent cluttering the notebook logs
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    # Wait for the service to start
    for i in range(15):
        if is_ollama_running():
            print("Ollama started successfully!")
            break
        time.sleep(1)
    else: 
        print("Warning: Ollama didn't respond on port 11434 after 15 seconds.")

# 3. Constants & Configuration

Before starting the generation, make sure your `config.yaml` is set up properly. To run efficiently on the Google Colab environment without crashing:
1. **Device**: Ensure `device: "cuda"` is set in your `config.yaml` (under `models`) to utilize the Colab GPU (T4/V100/A100) instead of CPU.
2. **RAM limit prevention**: Set `delete_after_run: true` in your `config.yaml` (under `models`). This instructs Ollama to delete each model from disk space and unload it from memory after executing, keeping RAM below Colab's 12 GB threshold.

In [ ]:
# Let's inspect the current config.yaml to verify settings
with open("config.yaml", "r", encoding="utf-8") as f:
    config_content = f.read()
print(config_content)

# 4. Run Complete Experiment Pipeline

This section runs both steps consecutively:
1. First, `main.py` processes the dataset and creates the initial generation results JSON files.
2. Second, `src/test_judges.py` performs the LLM-as-a-Judge evaluations on those results.

In [ ]:
print("=== STEP 1: Running Generation Experiments (main.py) ===\n")
!python main.py --save-data

print("\n=== STEP 2: Running LLM-as-a-Judge Evaluations (test_judges.py) ===\n")
!python src/test_judges.py

# 5. Strategy Comparison & Plotting

The evaluation plots comparing strategies (`Singolo-Distinti`, `Singolo-Aggregati`, `Doppio`) and experiment performance (RAG, LLM-only, RAG vs LLM) are automatically generated and saved under the corresponding experiment subdirectories inside `results/`.

In [ ]:
import glob
print("Checking generated result folders and plots:")
for path in glob.glob("results/*"):
    print(path)